#### 대상기업 :

In [22]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from DATA.stock_invest_function import *


In [23]:
def calculate_correlation_between_dfs(df1, df2, start_date=None, end_date=None, method='pearson', min_periods=4):
    """
    두 개의 시계열 DataFrame의 상관관계를 계산하되, 유효 관측치가 min_periods보다 많을 경우만 수행

    Parameters:
    ...
    - min_periods (int): 최소 유효 데이터 수

    Returns:
    - pd.DataFrame: 상관계수 매트릭스
    """
    if start_date:
        df1 = df1[df1.index >= pd.to_datetime(start_date)]
        df2 = df2[df2.index >= pd.to_datetime(start_date)]
    if end_date:
        df1 = df1[df1.index <= pd.to_datetime(end_date)]
        df2 = df2[df2.index <= pd.to_datetime(end_date)]

    combined = pd.merge(df1, df2, left_index=True, right_index=True, how='inner', suffixes=('_firm', '_hs'))

    corr_matrix = pd.DataFrame(index=df1.columns, columns=df2.columns, dtype=float)

    for firm in df1.columns:
        for hs in df2.columns:
            x = combined[firm]
            y = combined[hs]
            valid = x.notna() & y.notna()
            if valid.sum() >= min_periods:
                corr_matrix.loc[firm, hs] = x[valid].corr(y[valid], method=method)
            else:
                corr_matrix.loc[firm, hs] = np.nan  # 또는 0

    return corr_matrix

def get_top_correlated_hscode(corr_matrix, symbol, top_n=5, threshold=None, ascending=False):
    """
    특정 기업(Symbol)에 대해 상관관계가 높은 HS 코드를 추출하는 함수

    Parameters:
    - corr_matrix (pd.DataFrame): Symbol x HS_Code 형태의 상관관계 행렬
    - symbol (str): 대상 Symbol (예: '000080')
    - top_n (int): 상위 N개 추출 (threshold와 함께 사용 시 무시될 수 있음)
    - threshold (float or None): 상관계수 하한값 (예: 0.5 이상만 보기). 설정 시 top_n보다 우선함
    - ascending (bool): 상관계수 기준 오름차순 정렬 여부 (기본값: False = 높은 값 우선)

    Returns:
    - pd.DataFrame: root_hs_code 및 상관계수를 포함한 상위 N개 HS 코드
    """

    if symbol not in corr_matrix.index:
        raise ValueError(f"Symbol '{symbol}' not found in correlation matrix.")

    symbol_corr = corr_matrix.loc[symbol].dropna()

    if threshold is not None:
        symbol_corr = symbol_corr[symbol_corr >= threshold]

    top_correlated = symbol_corr.sort_values(ascending=ascending).head(top_n)

    return top_correlated.reset_index().rename(columns={'index': 'root_hs_code', symbol: 'correlation'})

def get_top_correlated_symbols(corr_matrix, hs_code, top_n=5, threshold=None, ascending=False):
    """
    특정 HS 코드에 대해 상관관계가 높은 기업 Symbol을 추출하는 함수

    Parameters:
    - corr_matrix (pd.DataFrame): Symbol x HS_Code 형태의 상관관계 행렬
    - hs_code (str or int): 대상 HS 코드 (예: '151550')
    - top_n (int): 상위 N개 추출
    - threshold (float or None): 상관계수 하한값 (예: 0.5 이상만 보기)
    - ascending (bool): 정렬 방향 (False: 높은 상관 우선)

    Returns:
    - pd.DataFrame: symbol 및 correlation 정보를 담은 상위 N개 결과
    """

    if hs_code not in corr_matrix.columns:
        raise ValueError(f"HS code '{hs_code}' not found in correlation matrix columns.")

    hs_corr = corr_matrix[hs_code].dropna()

    if threshold is not None:
        hs_corr = hs_corr[hs_corr >= threshold]

    top_symbols = hs_corr.sort_values(ascending=ascending).head(top_n)

    return top_symbols.reset_index().rename(columns={'index': 'symbol', hs_code: 'correlation'})


In [24]:
db_info = {
    'host': get_db_host(),
    # 'host': '192.168.0.230',
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

# SQLAlchemy 엔진 생성
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

# 테이블 이름
table_name = 'target_hs_code'

# 고유한 hs_code 값 추출 쿼리 실행
query = f"SELECT DISTINCT hs_code FROM {table_name}"
unique_hs_codes_df = pd.read_sql(query, con=engine)
hs_codes  = unique_hs_codes_df['hs_code'].unique().tolist()

indicator = 'expDlr'

df_real = fetch_trade_data_multi_hscode(db_info, hs_codes, indicator)

# 분기 정보 추가
df_real['quarter'] = df_real['date'].dt.to_period('Q')

# 그룹별로 분기별 합산
df_quarterly = (
    df_real
    .groupby(['root_hs_code', 'quarter'])['value']
    .sum()
    .reset_index()
)

# 👉 분기 월말로 변환 (예: 2007Q1 → 2007-03-31)
df_quarterly['date'] = df_quarterly['quarter'].dt.to_timestamp(how='end')

# 👉 'quarter' 컬럼 제거
df_quarterly.drop(columns=['quarter'], inplace=True)

# 1단계: 문자열로 직접 변환하려면 to_datetime 이후에 바로 strftime
df_quarterly['date'] = pd.to_datetime(df_quarterly['date']).dt.strftime('%Y-%m-%d')

def create_yoy_growth_pivot(df_quarterly, start_date=None, end_date=None):
    """
    전년 동분기 대비 증가율을 pivot 형태로 변환하고 분석기간을 설정할 수 있는 함수

    Parameters:
    - df_quarterly (DataFrame): 'root_hs_code', 'date', 'yoy_growth' 포함된 데이터
    - start_date (str or None): 분석 시작일 (예: '2015-01-01')
    - end_date (str or None): 분석 종료일 (예: '2023-12-31')

    Returns:
    - pivot_df (DataFrame): 행: date, 열: root_hs_code, 값: yoy_growth
    """
    # Pivot
    pivot_df = df_quarterly.pivot(
        index='date',
        columns='root_hs_code',
        values='yoy_growth'
    ).sort_index()

    # inf 값 NaN 처리
    pivot_df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # 분석 기간 슬라이싱 (날짜가 문자열이면 datetime으로 변환)
    pivot_df.index = pd.to_datetime(pivot_df.index)

    if start_date:
        pivot_df = pivot_df[pivot_df.index >= pd.to_datetime(start_date)]
    if end_date:
        pivot_df = pivot_df[pivot_df.index <= pd.to_datetime(end_date)]

    return pivot_df


# 전년 동분기 값 (4개 분기 전 값) 계산
df_quarterly['yoy_value'] = (
    df_quarterly
    .sort_values(['root_hs_code', 'date'])
    .groupby('root_hs_code')['value']
    .shift(4)
)

# ❗ yoy_growth 계산
df_quarterly['yoy_growth'] = (
    (df_quarterly['value'] - df_quarterly['yoy_value']) / df_quarterly['yoy_value']
) * 100

quarterly_trade_data = create_yoy_growth_pivot(df_quarterly, start_date='2008-03', end_date='2025-03')

In [25]:
fs_df = fetch_table_data(db_info, "korea_fs_data")
fs_df.rename(columns={'Date': 'date'}, inplace=True)

# 1. indicator 필터링
target_indicator = '매출액(천원)'
filtered_df = fs_df[fs_df['indicator'] == target_indicator].copy()

# 2. 날짜 정제 및 정렬
filtered_df['date'] = pd.to_datetime(filtered_df['date'])
filtered_df.sort_values(by='date', inplace=True)

# 3. value 컬럼이 있는지 확인 및 타입 강제
if 'value' not in filtered_df.columns:
    raise KeyError("'value' 컬럼이 없습니다.")

filtered_df['value'] = pd.to_numeric(filtered_df['value'], errors='coerce')

# 4. 피벗 테이블 생성 (행: date, 열: Symbol, 값: value)
pivot_df = filtered_df.pivot_table(
    index='date',
    columns='symbol',
    values='value',
    aggfunc='first'  # 중복 방지
)

# 5. 전년 동분기 대비 변화율 계산 (4분기 전 대비)
fs_yoy_growth_df = pivot_df.pct_change(periods=4) * 100

✅ 'korea_fs_data' 테이블에서 5584577건의 데이터를 가져왔습니다.


In [27]:
correlation_result = calculate_correlation_between_dfs(
    fs_yoy_growth_df,
    quarterly_trade_data,
    start_date='2020-03-31',
    end_date='2025-03-31'
)

# 상위 몇 개 확인
correlation_result.head()

root_hs_code,121120,121221,151550,151590,170199,190230,190590,200599,200830,200899,...,903149,903180,903190,903289,940130,940199,940330,940540,950300,970191
symbol,,,,,,,,,,,,,,,,,,,,,
A000010,0.295863,0.264535,-0.405793,-0.047399,0.113347,-0.355935,-0.755107,-0.567381,-0.156667,-0.361840,...,0.756018,0.006147,-0.309853,-0.000400,-0.629688,-0.853721,0.395599,-0.834817,-0.091785,0.732864
A000020,0.070134,0.536061,-0.176234,0.693954,-0.474600,-0.022520,-0.438492,-0.563861,-0.330848,-0.510009,...,0.100434,0.184839,0.321508,-0.092314,-0.516051,0.429429,0.382900,-0.587345,0.428124,-0.086697
A000030,0.303578,0.276907,-0.412560,-0.037144,0.091884,-0.374004,-0.760138,-0.592994,-0.176446,-0.395749,...,0.747137,0.045215,-0.294605,0.003928,-0.643218,-0.864607,0.418791,-0.837847,-0.067112,0.749627
A000040,0.155914,-0.175293,0.271092,-0.015489,-0.241148,-0.221853,-0.048957,-0.073037,0.269785,0.135000,...,0.061795,-0.302213,-0.210581,-0.469398,-0.240429,-0.071856,0.010413,0.305215,-0.099375,0.252328
A000050,-0.096418,0.026816,0.037008,-0.056782,0.031239,0.109311,0.391903,-0.040745,-0.046109,-0.300901,...,-0.391744,0.678166,0.183529,-0.012355,-0.218923,0.376524,0.284617,0.244170,0.230996,-0.532924


In [28]:
correlation_result.to_csv(r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\한국상장사_수출데이터_상관계수_202507.csv", index=True, encoding="utf-8-sig")

# path = r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\한국상장사_수출데이터_상관계수.csv"
# path = r"C:\Users\MetaM\PycharmProjects\stock_forecast\DATA\한국상장사_수출데이터_상관계수.csv"
# correlation_result = pd.read_csv(path).set_index('symbol')

In [ ]:
correlation_result.loc['A140860']['8502']

In [8]:
top_hs_codes = get_top_correlated_hscode(
    corr_matrix=correlation_result,  # 이전에 만든 상관관계 행렬
    symbol ='A005930',
    top_n=50,
    threshold=0.2  # 선택사항
)

print(top_hs_codes)

   root_hs_code  correlation
0        382219     0.920330
1        854141     0.855103
2        390729     0.810131
3        853400     0.778243
4        381239     0.775774
5        851713     0.731500
6        854232     0.727447
7        852589     0.706480
8        940199     0.705301
9        848690     0.692380
10       710510     0.672179
11       854231     0.665677
12       340290     0.664048
13       290121     0.655063
14       340420     0.650252
15       391000     0.648950
16       600537     0.644156
17       390330     0.643512
18       293190     0.637091
19       722300     0.632182
20       741110     0.631640
21       740721     0.627692
22       722020     0.615347
23       740921     0.614670
24       854149     0.600168
25       390110     0.595049
26       760612     0.593868
27       760711     0.590201
28       852491     0.589253
29       900120     0.586815
30       390931     0.582669
31       390769     0.582398
32       390730     0.582024
33       85241

In [9]:
from pykrx import stock

# 1. ticker 리스트 불러오기
tickers = stock.get_market_ticker_list(market="ALL")

# 2. ticker와 name을 리스트로 만들기
data = []
for t in tickers:
    name = stock.get_market_ticker_name(t)
    data.append({
        'ticker': t,
        'name': name,
        'symbol': 'A' + t
    })

# 3. DataFrame으로 변환
company_name_df = pd.DataFrame(data, columns=['symbol', 'ticker', 'name'])


In [42]:
top_symbols = get_top_correlated_symbols(
    corr_matrix=correlation_result,
    hs_code= '854232',
    top_n=50,
    threshold=0.1  # 선택사항
)

print(top_symbols)

     symbol  correlation
0   A000660     0.954314
1   A072870     0.869624
2   A108860     0.833098
3   A183300     0.828235
4   A066930     0.816630
5   A092600     0.792335
6   A071950     0.787571
7   A005930     0.785266
8   A059090     0.782823
9   A045300     0.778339
10  A034220     0.768949
11  A092220     0.760767
12  A011040     0.759996
13  A000700     0.753768
14  A072990     0.750104
15  A020760     0.744334
16  A009150     0.744010
17  A060480     0.743059
18  A011150     0.739587
19  A091340     0.736453
20  A037460     0.735252
21  A036120     0.731731
22  A198080     0.731550
23  A133750     0.729957
24  A000240     0.729427
25  A000680     0.728469
26  A034950     0.724575
27  A091970     0.720854
28  A056700     0.716961
29  A057050     0.716768
30  A036090     0.715722
31  A300120     0.712122
32  A239340     0.710563
33  A131180     0.710317
34  A005440     0.708843
35  A065680     0.708522
36  A052260     0.702741
37  A053110     0.702549
38  A059100     0.701466


In [43]:
pd.merge(top_symbols, company_name_df, on='symbol', how = 'left')

,symbol,correlation,ticker,name
0,A000660,0.954314,000660,SK하이닉스
1,A072870,0.869624,072870,메가스터디
2,A108860,0.833098,108860,셀바스AI
3,A183300,0.828235,183300,코미코
4,A066930,0.816630,NaN,NaN
5,A092600,0.792335,092600,앤씨앤
6,A071950,0.787571,071950,코아스
7,A005930,0.785266,005930,삼성전자
8,A059090,0.782823,059090,미코
9,A045300,0.778339,045300,성우테크론


In [10]:
hscode = fetch_table_data(db_info, "target_hs_code")

hscode[hscode['hs_code'] == '854232']

✅ 'target_hs_code' 테이블에서 567건의 데이터를 가져왔습니다.


,hs_code
417,854232


In [24]:
name

'흥아해운'